In [1]:
import pandas as pd
import numpy as np

In [2]:
# Ler o arquivo
df = pd.read_csv("Documents/Agregados_por_setores_demografia_BR.csv", dtype={"CD_setor": str})

# Código do estado
COD_EST = "35"

# Filtrar setores que começam com "35" (São Paulo)
df_sp = df[df["CD_setor"].str.startswith(COD_EST)].copy()

# colunas alvo
cols = [f"V010{c}" for c in range(34, 42)]  # V01034 ... V01041

# 1) Normalizar strings (remover espaços) e substituir 'X' por 0
df_sp[cols] = df_sp[cols].astype(str).apply(lambda s: s.str.strip().replace('X', '0', regex=False))

# 2) Converter para numérico (tenta trocar vírgula por ponto caso exista) e forçar NaN -> 0
df_sp[cols] = df_sp[cols].apply(lambda s: pd.to_numeric(s.str.replace(',', '.'), errors='coerce')).fillna(0).astype(int)

# 3) Recalcular somas
df_sp["pop_15_59"] = df_sp[["V01034", "V01035", "V01036", "V01037", "V01038", "V01039"]].sum(axis=1)
df_sp["pop_60_plus"] = df_sp[["V01040", "V01041"]].sum(axis=1)

# 4) (Opcional) DataFrame final só com CD_setor e as somas
df_final = df_sp[["CD_setor", "pop_15_59", "pop_60_plus"]].copy()

# Mostrar algumas linhas para conferir
print(df_final.head())

C:\Users\Nazivon Santos\AppData\Local\Temp\ipykernel_13652\1044548951.py:2: DtypeWarning: Columns (1,2,3,4,9,18,25,32,33,34) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("Documents/Agregados_por_setores_demografia_BR.csv", dtype={"CD_setor": str})


               CD_setor  pop_15_59  pop_60_plus
259264  350010505000001        176           78
259265  350010505000002        384          202
259266  350010505000003        183          165
259267  350010505000004        296          213
259268  350010505000005        331          212


In [3]:
# Salvar o dataframe em um novo arquivo
df_final.to_csv(f"agregados_setores_faixa_etaria_{COD_EST}.csv", index=False)

In [4]:
# Código do município é composto dos primeiros 6 algorismos do código do setor
df_sp["CD_MUN"] = df_sp["CD_setor"].str.slice(0, 6)

In [5]:
# Agrupar os valores das colunas relevantes por município
cols = [f"V010{c}" for c in range(34, 42)]  # V01034 ... V01041

df_mun = df_sp.groupby("CD_MUN")[cols].sum().reset_index()

In [6]:
# Salvar o dataframe de municípios em um novo arquivo
df_mun.to_csv(f"agregados_faixas_etarias_mun_{COD_EST}.csv", index=False)

In [7]:
# Inserir no dataframe uma coluna com os códigos dos municípios
df_final["CD_MUN"] = df_final["CD_setor"].str.slice(0, 6)
df_final

,CD_setor,pop_15_59,pop_60_plus,CD_MUN
259264,350010505000001,176,78,350010
259265,350010505000002,384,202,350010
259266,350010505000003,183,165,350010
259267,350010505000004,296,213,350010
259268,350010505000005,331,212,350010
...,...,...,...,...
360187,355730305000048,38,18,355730
360188,355730305000050,405,153,355730
360189,355730305000051,171,62,355730
360190,355730305000052,261,24,355730


In [13]:
# Ler o arquivo
df = pd.read_csv("Downloads/A17_mun_sp.csv", dtype={"CD_MUN": str})

In [14]:
df_setores_novo = df_final.merge(
    df,
    on="CD_MUN",
    how="left"
)

In [15]:
df_s = df_setores_novo

In [16]:
df_s

,CD_setor,pop_15_59,pop_60_plus,CD_MUN,PROPORÇÃO SEM PLANO_15-59,PROPORÇÃO SEM PLANO_60+,Crescimento_pop_15-59,Crescimento_pop_60+
0,350010505000001,176,78,350010,0.955192,0.955192,1.011665,1.052791
1,350010505000002,384,202,350010,0.955192,0.955192,1.011665,1.052791
2,350010505000003,183,165,350010,0.955192,0.955192,1.011665,1.052791
3,350010505000004,296,213,350010,0.955192,0.955192,1.011665,1.052791
4,350010505000005,331,212,350010,0.955192,0.955192,1.011665,1.052791
...,...,...,...,...,...,...,...,...
100923,355730305000048,38,18,355730,0.968990,0.968990,1.006721,1.077507
100924,355730305000050,405,153,355730,0.968990,0.968990,1.006721,1.077507
100925,355730305000051,171,62,355730,0.968990,0.968990,1.006721,1.077507
100926,355730305000052,261,24,355730,0.968990,0.968990,1.006721,1.077507


In [17]:
# Nova coluna 1 — população efetiva 15–59
df_s["pop_15_59_ajustada"] = (
    df_s["pop_15_59"] *
    df_s["Crescimento_pop_15-59"] *
    df_s["PROPORÇÃO SEM PLANO_15-59"]
)

# Nova coluna 2 — população efetiva 60+
df_s["pop_60_plus_ajustada"] = (
    df_s["pop_60_plus"] *
    df_s["Crescimento_pop_60+"] *
    df_s["PROPORÇÃO SEM PLANO_60+"]
)

In [18]:
df_n = df_s[["CD_setor", "pop_15_59_ajustada", "pop_60_plus_ajustada"]].copy()

In [19]:
df_s.to_csv(f"populacao_ajustada_por_setor_{COD_EST}.csv", index=False)

In [20]:
# Coeficientes - provindos dos Parâmetros do SUS (taxa de internação x tempo médio de permanência)
coef_15_59_clinicos = 0.0138 * 6.5
coef_15_59_cirurgicos = 0.0215 * 3.6

coef_60_clinicos = 0.0724 * 7.4
coef_60_cirurgicos = 0.044 * 4.6

In [21]:
# Criação das quatro colunas
df_n["leitos_clinicos_15_59"] = df_n["pop_15_59_ajustada"] * coef_15_59_clinicos
df_n["leitos_cirurgicos_15_59"] = df_n["pop_15_59_ajustada"] * coef_15_59_cirurgicos

df_n["leitos_clinicos_60+"] = df_n["pop_60_plus_ajustada"] * coef_60_clinicos
df_n["leitos_cirurgicos_60+"] = df_n["pop_60_plus_ajustada"] * coef_60_cirurgicos

In [22]:
# ρ (taxa média de ocupação) considerada nos parâmetros do SUS como 0.72
df_n["necessidade_total_leitos"] = (
    df_n["leitos_clinicos_15_59"]
    + df_n["leitos_cirurgicos_15_59"]
    + df_n["leitos_clinicos_60+"]
    + df_n["leitos_cirurgicos_60+"]
) / (365 * 0.72)

In [23]:
df_n.to_csv(f"setores_populacao_ajustada_{COD_EST}.csv", index=False)